# Creation of nice dataset for DPO experiment 1

## Loading Mistral 7b and the data

In [1]:
# Clone the repo (skip if already cloned)
!git clone https://github.com/JeremieTarantop/Cardiac-Diagnostic-CMR-report-.git repo_cmr
%cd repo_cmr

Cloning into 'repo_cmr'...
remote: Enumerating objects: 15639, done.
remote: Counting objects: 100% (15639/15639), done.
remote: Compressing objects: 100% (14891/14891), done.
remote: Total 15639 (delta 886), reused 14930 (delta 209), pack-reused 0 (from 0)
Receiving objects: 100% (15639/15639), 34.19 MiB | 40.76 MiB/s, done.
Resolving deltas: 100% (886/886), done.
/content/repo_cmr


In [2]:
from pathlib import Path

REPO = Path(".")
MEETI_TXT_ROOT = REPO / "data" / "MEETI TXT"

OUT_ROOT = REPO / "data"
OUT_A1 = OUT_ROOT / "artifact_1_no_ecg"
OUT_META = OUT_ROOT / "metadata"

for p in [OUT_A1, OUT_META]:
    p.mkdir(parents=True, exist_ok=True)

print("MEETI TXT root:", MEETI_TXT_ROOT)
print("Output root:", OUT_ROOT)


MEETI TXT root: data/MEETI TXT
Output root: data


In [13]:
import json

def safe_read_text(path: Path) -> str | None:
    try:
        return path.read_text(encoding="utf-8").strip()
    except UnicodeDecodeError:
        # fallback if weird encoding slipped in
        try:
            return path.read_text(encoding="latin-1").strip()
        except Exception:
            return None
    except Exception:
        return None

# def collect_meeti_records(meeti_root: Path, limit: int = 500):
#     report_files = sorted(meeti_root.rglob("*_report.txt"))
#     records = []
#     skipped = 0

#     for rf in report_files:
#         rec_id = rf.name.replace("_report.txt", "")
#         interp_f = rf.parent / f"{rec_id}_llm_interpretation.txt"

#         short_report = safe_read_text(rf)
#         llm_interp = safe_read_text(interp_f) if interp_f.exists() else None

#         if not short_report or not llm_interp:
#             skipped += 1
#             continue

#         records.append({
#             "id": rec_id,
#             "short_report": short_report,
#             "llm_interpretation": llm_interp,
#             "folder": str(rf.parent),
#             "report_path": str(rf),
#             "interpretation_path": str(interp_f),
#         })

#         if len(records) >= limit:
#             break

#     return records, {"found_report_files": len(report_files), "loaded": len(records), "skipped": skipped}

# records, stats = collect_meeti_records(MEETI_TXT_ROOT, limit=100)
# stats

def collect_meeti_records(meeti_root: Path, limit: int = 500, skip: int = 0):
    report_files = sorted(meeti_root.rglob("*_report.txt"))
    records = []

    skipped_invalid = 0
    skipped_valid = 0

    first_skipped_id = None
    last_skipped_id = None

    for rf in report_files:
        rec_id = rf.name.replace("_report.txt", "")
        interp_f = rf.parent / f"{rec_id}_llm_interpretation.txt"

        short_report = safe_read_text(rf)
        llm_interp = safe_read_text(interp_f) if interp_f.exists() else None

        # Skip invalid files
        if not short_report or not llm_interp:
            skipped_invalid += 1
            continue

        # Skip first `skip` valid ones
        if skipped_valid < skip:
            if first_skipped_id is None:
                first_skipped_id = rec_id
            last_skipped_id = rec_id
            skipped_valid += 1
            continue

        records.append({
            "id": rec_id,
            "short_report": short_report,
            "llm_interpretation": llm_interp,
            "folder": str(rf.parent),
            "report_path": str(rf),
            "interpretation_path": str(interp_f),
        })

        if len(records) >= limit:
            break

    stats = {
        "found_report_files": len(report_files),
        "loaded": len(records),
        "skipped_invalid": skipped_invalid,
        "skipped_initial_valid": skipped_valid,
        "first_skipped_valid_id": first_skipped_id,
        "last_skipped_valid_id": last_skipped_id,
    }

    return records, stats

records, stats = collect_meeti_records(MEETI_TXT_ROOT,limit=200,skip=100)

print(stats)

{'found_report_files': 5000, 'loaded': 200, 'skipped_invalid': 0, 'skipped_initial_valid': 100, 'first_skipped_valid_id': '40689238', 'last_skipped_valid_id': '46566322'}


In [14]:
from pathlib import Path

ROOT = Path(".")
MEETI_TXT_DIR = ROOT / "data" / "MEETI TXT"

PROMPT_PATH = ROOT / "prompt_v1.txt"
assert PROMPT_PATH.exists(), f"Missing: {PROMPT_PATH}"
assert MEETI_TXT_DIR.exists(), f"Missing: {MEETI_TXT_DIR}"

BASE_PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()

OUT_DIR = ROOT / "data" / "outputs_dpo_mistral" / "artifacts_1_2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OK")
print("PROMPT_PATH:", PROMPT_PATH)
print("MEETI_TXT_DIR:", MEETI_TXT_DIR)
print("OUT_DIR:", OUT_DIR)


OK
PROMPT_PATH: prompt_v1.txt
MEETI_TXT_DIR: data/MEETI TXT
OUT_DIR: data/outputs_dpo_mistral/artifacts_1_2


In [5]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit config (recommended on Colab)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Model (generator)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_cfg,
    torch_dtype=torch.float16,
)
model.eval()

print("Loaded:", MODEL_ID)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.3


### Generating the report

In [15]:
@torch.inference_mode()
def mistral_generate_from_prompt(user_prompt: str, max_new_tokens: int = 600, temperature: float = 0.7, top_p: float = 0.95) -> str:
    messages = [
        {"role": "system", "content": "Follow the user instructions exactly."},
        {"role": "user", "content": user_prompt},
    ]
    chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded[len(chat):].strip() if decoded.startswith(chat) else decoded.strip()


In [17]:
from pathlib import Path

def gen_artifact1_for_patient(
    rec: dict,
    prompt: str = BASE_PROMPT,
    output_dir: Path = Path("generated_reports")
) -> Path:

    output_dir.mkdir(exist_ok=True)

    rec_id = rec["id"]

    # Generate hallucinated report
    text = mistral_generate_from_prompt(prompt)

    # Save in central folder (NOT inside MEETI dataset)
    out_path = output_dir / f"{rec_id}_hallucinated.txt"

    out_path.write_text(text + "\n", encoding="utf-8")

    return out_path

In [ ]:
# from pathlib import Path

# for rec in records:
#     path = gen_artifact1_for_patient(rec)
#     print("Saved:", path)

Saved: generated_reports/40689238_hallucinated.txt
Saved: generated_reports/44458630_hallucinated.txt
Saved: generated_reports/49036311_hallucinated.txt
Saved: generated_reports/45090959_hallucinated.txt
Saved: generated_reports/48446569_hallucinated.txt
Saved: generated_reports/42709053_hallucinated.txt
Saved: generated_reports/41445586_hallucinated.txt
Saved: generated_reports/42695383_hallucinated.txt
Saved: generated_reports/40067704_hallucinated.txt
Saved: generated_reports/42947358_hallucinated.txt
Saved: generated_reports/43522917_hallucinated.txt
Saved: generated_reports/44095784_hallucinated.txt
Saved: generated_reports/45386375_hallucinated.txt
Saved: generated_reports/48339811_hallucinated.txt
Saved: generated_reports/40539087_hallucinated.txt
Saved: generated_reports/43681375_hallucinated.txt
Saved: generated_reports/44069449_hallucinated.txt
Saved: generated_reports/44837313_hallucinated.txt
Saved: generated_reports/47218930_hallucinated.txt
Saved: generated_reports/406952

In [ ]:
!zip -r generated_reports.zip generated_reports

  adding: generated_reports/ (stored 0%)
  adding: generated_reports/41366957_hallucinated.txt (deflated 61%)
  adding: generated_reports/45808859_hallucinated.txt (deflated 62%)
  adding: generated_reports/44837313_hallucinated.txt (deflated 63%)
  adding: generated_reports/49245181_hallucinated.txt (deflated 62%)
  adding: generated_reports/45594411_hallucinated.txt (deflated 62%)
  adding: generated_reports/47499371_hallucinated.txt (deflated 63%)
  adding: generated_reports/44458630_hallucinated.txt (deflated 63%)
  adding: generated_reports/40067704_hallucinated.txt (deflated 62%)
  adding: generated_reports/40695233_hallucinated.txt (deflated 62%)
  adding: generated_reports/43681375_hallucinated.txt (deflated 62%)
  adding: generated_reports/42546971_hallucinated.txt (deflated 63%)
  adding: generated_reports/41798696_hallucinated.txt (deflated 61%)
  adding: generated_reports/41420867_hallucinated.txt (deflated 62%)
  adding: generated_reports/41309390_hallucinated.txt (deflate

In [ ]:
# from google.colab import files
# files.download("generated_reports.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# # Run on first patient
# rec0 = records[0]
# print("Testing ID:", rec0["id"])
# print("Leaf folder:", rec0["folder"])

# a1_path = gen_artifact1_for_patient(rec0)

# print("Wrote artifact 1 to:", a1_path)


Testing ID: 40689238
Leaf folder: data/MEETI TXT/p1000/p10000032/s40689238
Wrote artifact 1 to: data/MEETI TXT/p1000/p10000032/s40689238/40689238_mistral_hallucination.txt


### Generating reports with a little bit of ECG info (ECG report)

In [19]:
from pathlib import Path

# Where to save the new generations (separate folder from the prompt-only ones)
GEN_WITH_REPORT_DIR = ROOT / "data" / "20260209_MEETI TXT" / "generated_reports_with_report"
GEN_WITH_REPORT_DIR.mkdir(parents=True, exist_ok=True)

def build_prompt_with_report(base_prompt: str, meeti_report_text: str) -> str:
    """
    Keep the SAME base prompt, but add the MEETI machine-style summary (the *_report.txt content)
    as extra input context.
    """
    return (
        base_prompt.strip()
        + "\n\n"
        + "Additional case information. Do not mention it in the CMR report though. Just use it as external information. (ECG summary from dataset):\n"
        + meeti_report_text.strip()
        + "\n\n"
        + "Now write the report following the same instructions."
    )

def generate_reports_with_meeti_report_input(
    records: list[dict],
    n_files: int = 100,
    out_dir: Path = GEN_WITH_REPORT_DIR,
    max_new_tokens: int = 1000,
    temperature: float = 0.7,
    top_p: float = 0.95,
) -> list[Path]:
    """
    Generates n_files reports by conditioning on the MEETI *_report.txt content (rec["short_report"]).
    Saves outputs to out_dir with filenames: {id}_with_report.txt
    Returns list of saved paths.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    for rec in records[:n_files]:
        rec_id = rec["id"]

        # This is the content of PatientID_report.txt (already loaded into your records as short_report)
        meeti_summary = rec["short_report"]

        # Build the actual prompt sent to Mistral
        user_prompt = build_prompt_with_report(BASE_PROMPT, meeti_summary)

        # Generate
        text = mistral_generate_from_prompt(
            user_prompt=user_prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )

        # Save
        out_path = out_dir / f"{rec_id}_with_report.txt"
        out_path.write_text(text + "\n", encoding="utf-8")
        saved.append(out_path)

    return saved



In [23]:
# ---- Run (change n_files anytime) ----
# Make sure 'records' is already loaded, e.g. records, stats = collect_meeti_records(MEETI_TXT_ROOT, limit=500)
n_files = 200
paths = generate_reports_with_meeti_report_input(records=records, n_files=n_files)

print(f"Generated {len(paths)} files in: {GEN_WITH_REPORT_DIR}")
print("Example:", paths[0] if paths else None)

Generated 200 files in: data/20260209_MEETI TXT/generated_reports_with_report
Example: data/20260209_MEETI TXT/generated_reports_with_report/40288161_with_report.txt


In [21]:
from pathlib import Path

def show_first_n_generated_with_short(records, gen_dir, n=2):
    for i, rec in enumerate(records[:n], start=1):
        rec_id = rec["id"]
        gen_path = gen_dir / f"{rec_id}_with_report.txt"

        if not gen_path.exists():
            print(f"[{i}] Missing generated file for {rec_id}")
            continue

        generated_text = gen_path.read_text(encoding="utf-8").strip()

        print("=" * 100)
        print(f"[{i}] ID: {rec_id}")
        print("=" * 100)

        print("\n--- SHORT ECG REPORT (input) ---")
        print(rec["short_report"])

        print("\n--- GENERATED REPORT (with ECG summary) ---")
        print(generated_text)

        print("\n\n")

# Run it
show_first_n_generated_with_short(
    records=records,
    gen_dir=GEN_WITH_REPORT_DIR,
    n=3
)

[1] ID: 40288161

--- SHORT ECG REPORT (input) ---
Atrial fibrillation with PVCs.; Left bundle branch block; Abnormal ECG.

--- GENERATED REPORT (with ECG summary) ---
Follow the user instructions exactly.

A.1 Diagnosis Guider Prompt 
# Your task:  Interpret the provided ECG data, identify key features and abnormalities in each lead, and generate a clinical diagnosis that is supported by the observed evidence. 

## Key objectives:
1.  Simulate a Realistic Diagnostic Process:  The interpretation should reflect how a doctor would analyze an ECG, ask clarifying questions, and arrive at a diagnosis. 
2.  Grounded ECG Understanding:  The analysis should be based on specific ECG features and explicitly reference these features as evidence. 
3.  Evidence-Based Reasoning:  The diagnosis should be supported by clear, logical reasoning tied to the ECG findings. 

## Guidelines for the ECG analysis: 
1.  Data: ECG embeddings generated by PCLR
2.  Act as a cardiologist and use medical knowledge t

In [24]:
import shutil
from pathlib import Path

# Folder that contains your generated files
FOLDER_TO_ZIP = GEN_WITH_REPORT_DIR  # already defined in your notebook

ZIP_PATH = Path("generated_reports_with_report.zip")

# Create zip
shutil.make_archive(
    base_name=ZIP_PATH.with_suffix(""),
    format="zip",
    root_dir=FOLDER_TO_ZIP,
)

print("Zip created at:", ZIP_PATH.resolve())

Zip created at: /content/repo_cmr/generated_reports_with_report.zip


In [25]:
from google.colab import files

files.download("generated_reports_with_report.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Adding files directly into the github

In [ ]:
!git config --global user.email "tarantojeremie@gmail.com"
!git config --global user.name "JeremieTarantop"